# Create Swedish Collegium for Advanced Study Former Fellows Awards

Creates OpenAlex award rows from the Swedish Collegium for Advanced Study (SCAS) official all-former-fellows page.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/scas_former_fellows_to_s3.py` to download the official page, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes all parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** `https://www.swedishcollegium.se/fellows/former-fellows/all-former-fellows`  
**S3 location:** `s3a://openalex-ingest/awards/scas/scas_former_fellows.parquet`  
**Local full-source validation on 2026-05-28:** 784 award rows from the official SCAS all-former-fellows page after splitting repeated terms; 100% native ID/name/term/scheme/display-name/landing-page coverage; 783/784 rows (99.9%) with affiliation; 782/784 rows (99.7%) with parsed years; years 1986-2025; 764 Fellow-in-Residence rows and 20 Short-Term Researcher rows.

**SCAS funder:**
- funder_id: 4320319588
- display_name: "Swedish Collegium for Advanced Study"
- country: SE

**Source-shape note:** The source page also lists "Former Associated Researchers". This ingest excludes that section because the page does not identify those rows as fellowships or award-style appointments. Current fellows and dedicated Pro Futura pages are left for a possible follow-up source expansion.

**Amount/currency decision:** Amount and currency are NULL by runbook section 6.7 waiver because SCAS does not publish per-fellowship monetary amounts on the official listing. This follows the fellowship/prize waiver pattern used for HHMI, CIFAR, and Rome Prize-style fellowships.

**Mapping summary:** One OpenAlex award row per SCAS fellow/researcher term. Native key is a stable `scas-{section}-{year-range}-{name}-{term}-{hash}` identifier derived from the official section, name, affiliation, and term. The fellow is mapped to `lead_investigator`; source affiliation is mapped to `lead_investigator.affiliation.name`.

## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.scas_former_fellows_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/scas/scas_former_fellows.parquet`;

In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-28 found 784 rows.
SELECT COUNT(*) as total_scas_former_fellow_awards
FROM openalex.awards.scas_former_fellows_raw;

In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.scas_former_fellows_raw;

In [ ]:
%sql
-- Sample raw SCAS data.
SELECT
    funder_award_id,
    source_section,
    scheme_label,
    name,
    affiliation,
    term,
    start_year,
    end_year,
    profile_url,
    landing_page_url
FROM openalex.awards.scas_former_fellows_raw
LIMIT 10;

## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'scas_former_fellows_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|obligated|outlay|grant|award|donation|support|stipend|fellowship';

In [ ]:
%sql
-- Confirm source coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(funder_award_id) AS rows_with_native_id,
    COUNT(display_name) AS rows_with_display_name,
    COUNT(name) AS rows_with_name,
    COUNT(affiliation) AS rows_with_affiliation,
    ROUND(try_divide(COUNT(affiliation), COUNT(*)) * 100.0, 1) AS pct_affiliation,
    COUNT(term) AS rows_with_term,
    COUNT(start_year) AS rows_with_start_year,
    ROUND(try_divide(COUNT(start_year), COUNT(*)) * 100.0, 1) AS pct_start_year,
    COUNT(landing_page_url) AS rows_with_landing_page_url,
    COUNT(amount) AS rows_with_amount,
    MIN(TRY_CAST(start_year AS INT)) AS min_start_year,
    MAX(TRY_CAST(end_year AS INT)) AS max_end_year
FROM openalex.awards.scas_former_fellows_raw;

In [ ]:
%sql
-- Native-key inspection. funder_award_id must be unique.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT source_record_id) AS distinct_source_record_ids,
    COUNT(DISTINCT CONCAT(source_section, ':', name, ':', term)) AS distinct_section_name_terms
FROM openalex.awards.scas_former_fellows_raw;

In [ ]:
%sql
-- Scheme and source-section distribution.
SELECT source_section, scheme_label, funding_type, COUNT(*) AS cnt
FROM openalex.awards.scas_former_fellows_raw
GROUP BY source_section, scheme_label, funding_type
ORDER BY cnt DESC;

In [ ]:
%sql
-- Year distribution and the small no-year source edge cases.
SELECT
    COALESCE(start_year, 'NO_YEAR_IN_SOURCE_TERM') AS start_year_bucket,
    COUNT(*) AS cnt
FROM openalex.awards.scas_former_fellows_raw
GROUP BY COALESCE(start_year, 'NO_YEAR_IN_SOURCE_TERM')
ORDER BY start_year_bucket DESC;

In [ ]:
%sql
-- Rows with explicit source NULLs retained from the official listing.
SELECT name, affiliation, term, source_section, landing_page_url
FROM openalex.awards.scas_former_fellows_raw
WHERE affiliation IS NULL OR start_year IS NULL
ORDER BY name, term;

## Step 1.6: Funder Existence Fail-Fast

In [ ]:
%sql
-- Must return TRUE. If this fails, STOP: SCAS is missing from openalex.common.funder.
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for Swedish Collegium for Advanced Study F4320319588'
) AS funder_row_exists
FROM openalex.common.funder
WHERE funder_id = 4320319588;

In [ ]:
%sql
-- Inspect the funder row used for the award struct.
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320319588;

## Step 2: Transform to OpenAlex Award Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.scas_former_fellows_awards
USING delta
AS
WITH
scas_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320319588
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        CASE
            WHEN NULLIF(TRIM(currency), '') IS NULL THEN NULL
            ELSE UPPER(TRIM(currency))
        END AS parsed_currency,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(start_year AS INT) AS parsed_start_year,
        TRY_CAST(end_year AS INT) AS parsed_end_year
    FROM openalex.awards.scas_former_fellows_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
      AND name IS NOT NULL
      AND TRIM(name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,

        TRIM(g.display_name) as display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END as description,

        f.funder_id,
        g.native_award_id as funder_award_id,

        g.parsed_amount as amount,
        g.parsed_currency as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        NULLIF(TRIM(g.funding_type), '') as funding_type,
        NULLIF(TRIM(g.scheme_label), '') as funder_scheme,

        'scas_former_fellows' as provenance,

        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        g.parsed_start_year as start_year,
        g.parsed_end_year as end_year,

        struct(
            NULLIF(TRIM(g.given_name), '') as given_name,
            NULLIF(TRIM(g.family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                NULLIF(TRIM(g.affiliation), '') as name,
                CAST(NULL AS STRING) as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN scas_funder f
)

SELECT * FROM awards_transformed;

## Step 3: Delete Previous Source Rows and Insert Priority 174

In [ ]:
%sql
-- Remove previous SCAS former-fellows data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'scas_former_fellows' AND priority = 174;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    174 as priority
FROM openalex.awards.scas_former_fellows_awards;

## Step 6: Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_scas_former_fellow_awards
FROM openalex.awards.scas_former_fellows_awards;

In [ ]:
%sql
-- Confirm the transformed table has the OpenAlex awards columns only.
DESCRIBE openalex.awards.scas_former_fellows_awards;

In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator.given_name AS given_name,
    lead_investigator.family_name AS family_name,
    lead_investigator.affiliation.name AS source_affiliation,
    landing_page_url
FROM openalex.awards.scas_former_fellows_awards
LIMIT 10;

In [ ]:
%sql
-- Check ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.scas_former_fellows_awards;

In [ ]:
%sql
-- Check funder distribution. Should be only SCAS.
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.scas_former_fellows_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;

In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(start_year) as has_start_year,
    COUNT(end_year) as has_end_year,
    COUNT(landing_page_url) as has_url,
    COUNT(lead_investigator.given_name) as has_given_name,
    COUNT(lead_investigator.family_name) as has_family_name,
    COUNT(lead_investigator.affiliation.name) as has_source_affiliation,
    ROUND(try_divide(COUNT(start_year), COUNT(*)) * 100.0, 1) as pct_start_year,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name), COUNT(*)) * 100.0, 1) as pct_source_affiliation,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount
FROM openalex.awards.scas_former_fellows_awards;

In [ ]:
%sql
-- Amount/currency waiver check. SCAS listing does not publish per-fellowship amounts.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    COUNT(currency) AS rows_with_currency,
    collect_set(currency) AS currencies
FROM openalex.awards.scas_former_fellows_awards;

In [ ]:
%sql
-- Year distribution.
SELECT COALESCE(CAST(start_year AS STRING), 'NO_YEAR_IN_SOURCE_TERM') AS start_year_bucket, COUNT(*) as cnt
FROM openalex.awards.scas_former_fellows_awards
GROUP BY COALESCE(CAST(start_year AS STRING), 'NO_YEAR_IN_SOURCE_TERM')
ORDER BY start_year_bucket DESC;

In [ ]:
%sql
-- Funding type and scheme distribution.
SELECT funding_type, funder_scheme, COUNT(*) as cnt
FROM openalex.awards.scas_former_fellows_awards
GROUP BY funding_type, funder_scheme
ORDER BY cnt DESC;

In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 174.
SELECT
    priority,
    provenance,
    COUNT(*) as cnt,
    COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids,
    COUNT(amount) as rows_with_amount
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'scas_former_fellows' AND priority = 174
GROUP BY priority, provenance;

## Handoff / Admin Notes

Contractor-complete handoff only. The contractor has no S3 or Databricks access, so an admin must:

1. Run `scripts/local/scas_former_fellows_to_s3.py --allow-shrink` only if the section 1.4 shrink check is intentionally overridden; otherwise run without that flag.
2. Upload `s3://openalex-ingest/awards/scas/scas_former_fellows.parquet` via the script.
3. Run this notebook on Databricks.
4. Run the Step 6 verification cells and QA the inserted `provenance='scas_former_fellows'`, `priority=174` rows.
5. Mark the tracker row Complete after production API verification.

Do not add job YAML until an admin has run and verified the ingest.